In [ ]:
%pip install openai mlflow pyyaml

# PT with PPT Fallback Router

Defines a `PTWithPPTFallback` MLflow pyfunc model that routes every request to a provisioned
throughput endpoint first. On capacity errors (429, 503) it retries immediately against the
equivalent pay-per-token model — the fallback is transparent to the caller.

Configure `config.yml` before running:
- `databricks.host` — your workspace URL
- `model_serving.provisioned_endpoint` — name of your deployed PT endpoint
- `model_serving.pay_per_token_model` — the same underlying model available via pay-per-token

In [ ]:
import yaml

with open("config.yml") as f:
    config = yaml.safe_load(f)

DATABRICKS_HOST       = config["databricks"]["host"]
SECRETS_SCOPE         = config["databricks"]["secrets_scope"]
TOKEN_SECRET_KEY      = config["databricks"]["token_secret_key"]
PROVISIONED_ENDPOINT  = config["model_serving"]["provisioned_endpoint"]
PAY_PER_TOKEN_MODEL   = config["model_serving"]["pay_per_token_model"]
FALLBACK_STATUS_CODES = config["model_serving"]["fallback_status_codes"]

In [ ]:
import logging

import mlflow
from mlflow.pyfunc import PythonModel
from openai import APIStatusError, OpenAI, RateLimitError

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Model definition

In [ ]:
class PTWithPPTFallback(PythonModel):
    def load_context(self, context):
        cfg = context.model_config
        self.provisioned_endpoint  = cfg["provisioned_endpoint"]
        self.pay_per_token_model   = cfg["pay_per_token_model"]
        self.fallback_status_codes = set(cfg["fallback_status_codes"])
        self.client = OpenAI(
            base_url=f"{cfg['host']}/serving-endpoints",
            api_key=dbutils.secrets.get(scope=cfg["secrets_scope"], key=cfg["token_secret_key"]),
        )

    def predict(self, context, model_input, params=None):
        messages = model_input["messages"]
        params   = params or {}
        try:
            response = self.client.chat.completions.create(
                model=self.provisioned_endpoint,
                messages=messages,
                **params,
            )
            logger.info("route=provisioned endpoint=%s", self.provisioned_endpoint)
            return response
        except (RateLimitError, APIStatusError) as e:
            if getattr(e, "status_code", None) not in self.fallback_status_codes:
                raise
            logger.warning(
                "route=pay_per_token model=%s reason=fallback status=%s",
                self.pay_per_token_model,
                e.status_code,
            )
            return self.client.chat.completions.create(
                model=self.pay_per_token_model,
                messages=messages,
                **params,
            )

## Log model

In [ ]:
with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        artifact_path="pt_ppt_router",
        python_model=PTWithPPTFallback(),
        model_config={
            "host":                  DATABRICKS_HOST,
            "secrets_scope":         SECRETS_SCOPE,
            "token_secret_key":      TOKEN_SECRET_KEY,
            "provisioned_endpoint":  PROVISIONED_ENDPOINT,
            "pay_per_token_model":   PAY_PER_TOKEN_MODEL,
            "fallback_status_codes": FALLBACK_STATUS_CODES,
        },
    )
    print(f"Logged: {model_info.model_uri}")

## Load and use

In [ ]:
router = mlflow.pyfunc.load_model(model_info.model_uri)

In [ ]:
# Non-streaming
response = router.predict(
    {"messages": [{"role": "user", "content": "What is the capital of France?"}]}
)
print(response.choices[0].message.content)

In [ ]:
# Streaming
stream = router.predict(
    {"messages": [{"role": "user", "content": "Explain provisioned throughput in one paragraph."}]},
    params={"stream": True},
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)